# 01 - Seed Data Collection

This notebook collects parallel corpora from multiple sources for the Daraja translation pipeline.

**Sources:**
- OPUS corpora (JW300, WikiMatrix, CCAligned)
- Flores-200 benchmark data
- NLLB parallel data

**Target Language Pairs:**
1. Somali (so) → Swahili (sw) - Primary
2. Tigrinya (ti) → Arabic (ar)
3. Dari (prs) → Turkish (tr)

## Setup

In [ ]:
# Install dependencies
!pip install -q datasets opustools pandas tqdm

In [ ]:
import os
import sys
from pathlib import Path
from collections import defaultdict
import json

import pandas as pd
from tqdm.auto import tqdm

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data.seed_loader import SeedDataLoader, ParallelSentence, CorpusStats

# Configuration
DATA_DIR = Path('../data/seed')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")

## Language Pair Configuration

In [ ]:
LANGUAGE_PAIRS = [
    {
        'name': 'Somali-Swahili',
        'source': 'so',
        'target': 'sw',
        'priority': 1,
        'corpora': ['jw300', 'flores200']
    },
    {
        'name': 'Tigrinya-Arabic',
        'source': 'ti',
        'target': 'ar',
        'priority': 2,
        'corpora': ['jw300', 'flores200']
    },
    {
        'name': 'Dari-Turkish',
        'source': 'prs',
        'target': 'tr',
        'priority': 3,
        'corpora': ['jw300', 'flores200']
    }
]

print("Target language pairs:")
for pair in LANGUAGE_PAIRS:
    print(f"  {pair['priority']}. {pair['name']} ({pair['source']} → {pair['target']})")

## Load Flores-200 Data

In [ ]:
from datasets import load_dataset

# Flores language codes
FLORES_CODES = {
    'so': 'som_Latn',
    'sw': 'swh_Latn',
    'ti': 'tir_Ethi',
    'ar': 'arb_Arab',
    'prs': 'prs_Arab',
    'tr': 'tur_Latn',
}

def load_flores_pair(source_code: str, target_code: str, split: str = 'devtest'):
    """Load Flores-200 parallel data for a language pair."""
    flores_src = FLORES_CODES.get(source_code)
    flores_tgt = FLORES_CODES.get(target_code)
    
    if not flores_src or not flores_tgt:
        print(f"Warning: Flores codes not found for {source_code}-{target_code}")
        return []
    
    try:
        # Load both languages
        ds_src = load_dataset('facebook/flores', flores_src, split=split, trust_remote_code=True)
        ds_tgt = load_dataset('facebook/flores', flores_tgt, split=split, trust_remote_code=True)
        
        pairs = []
        for src_item, tgt_item in zip(ds_src, ds_tgt):
            pairs.append(ParallelSentence(
                source=src_item['sentence'],
                target=tgt_item['sentence'],
                source_lang=source_code,
                target_lang=target_code,
                corpus='flores200',
                metadata={'split': split}
            ))
        
        print(f"Loaded {len(pairs)} pairs from Flores-200 ({source_code}-{target_code})")
        return pairs
    except Exception as e:
        print(f"Error loading Flores-200: {e}")
        return []

In [ ]:
# Load Flores-200 for all pairs
flores_data = {}

for pair in LANGUAGE_PAIRS:
    key = f"{pair['source']}-{pair['target']}"
    flores_data[key] = load_flores_pair(pair['source'], pair['target'])
    
print(f"\nTotal Flores-200 pairs: {sum(len(v) for v in flores_data.values())}")

## Load OPUS Data (JW300)

In [ ]:
def load_opus_jw300(source_code: str, target_code: str, max_pairs: int = 50000):
    """Load JW300 parallel data from OPUS."""
    try:
        # Try HuggingFace OPUS loader
        from datasets import load_dataset
        
        pair_code = f"{source_code}-{target_code}"
        
        ds = load_dataset('opus100', pair_code, split='train', trust_remote_code=True)
        
        pairs = []
        for i, item in enumerate(ds):
            if i >= max_pairs:
                break
            pairs.append(ParallelSentence(
                source=item['translation'][source_code],
                target=item['translation'][target_code],
                source_lang=source_code,
                target_lang=target_code,
                corpus='opus_jw300'
            ))
        
        print(f"Loaded {len(pairs)} pairs from OPUS ({source_code}-{target_code})")
        return pairs
        
    except Exception as e:
        print(f"OPUS load failed for {source_code}-{target_code}: {e}")
        return []

In [ ]:
# Load OPUS data for all pairs
opus_data = {}

for pair in LANGUAGE_PAIRS:
    key = f"{pair['source']}-{pair['target']}"
    opus_data[key] = load_opus_jw300(pair['source'], pair['target'])

print(f"\nTotal OPUS pairs: {sum(len(v) for v in opus_data.values())}")

## Combine and Clean Data

In [ ]:
def clean_and_combine(opus_pairs, flores_pairs):
    """Combine and deduplicate parallel pairs."""
    all_pairs = opus_pairs + flores_pairs
    
    # Deduplicate
    seen = set()
    unique_pairs = []
    
    for pair in all_pairs:
        key = (pair.source.strip().lower(), pair.target.strip().lower())
        if key not in seen and len(pair.source) >= 5 and len(pair.target) >= 5:
            seen.add(key)
            unique_pairs.append(pair)
    
    return unique_pairs

In [ ]:
# Combine all data
combined_data = {}

for pair in LANGUAGE_PAIRS:
    key = f"{pair['source']}-{pair['target']}"
    combined_data[key] = clean_and_combine(
        opus_data.get(key, []),
        flores_data.get(key, [])
    )
    print(f"{pair['name']}: {len(combined_data[key])} unique pairs")

## Compute Statistics

In [ ]:
def compute_corpus_stats(pairs):
    """Compute statistics for a list of parallel pairs."""
    if not pairs:
        return None
    
    source_tokens = 0
    target_tokens = 0
    source_vocab = set()
    target_vocab = set()
    
    for p in pairs:
        src_words = p.source.split()
        tgt_words = p.target.split()
        source_tokens += len(src_words)
        target_tokens += len(tgt_words)
        source_vocab.update(src_words)
        target_vocab.update(tgt_words)
    
    return {
        'total_pairs': len(pairs),
        'source_tokens': source_tokens,
        'target_tokens': target_tokens,
        'source_vocab': len(source_vocab),
        'target_vocab': len(target_vocab),
        'avg_source_len': source_tokens / len(pairs),
        'avg_target_len': target_tokens / len(pairs),
    }

In [ ]:
# Compute and display stats
all_stats = {}

for pair in LANGUAGE_PAIRS:
    key = f"{pair['source']}-{pair['target']}"
    stats = compute_corpus_stats(combined_data.get(key, []))
    if stats:
        all_stats[key] = stats
        print(f"\n{pair['name']} Statistics:")
        print(f"  Total pairs: {stats['total_pairs']:,}")
        print(f"  Source vocab: {stats['source_vocab']:,}")
        print(f"  Target vocab: {stats['target_vocab']:,}")
        print(f"  Avg source length: {stats['avg_source_len']:.1f} tokens")
        print(f"  Avg target length: {stats['avg_target_len']:.1f} tokens")

## Save Data

In [ ]:
def save_parallel_files(pairs, output_dir, prefix):
    """Save parallel pairs to separate files."""
    if not pairs:
        return
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    source_lang = pairs[0].source_lang
    target_lang = pairs[0].target_lang
    
    source_file = output_dir / f"{prefix}.{source_lang}"
    target_file = output_dir / f"{prefix}.{target_lang}"
    
    with open(source_file, 'w', encoding='utf-8') as sf:
        with open(target_file, 'w', encoding='utf-8') as tf:
            for p in pairs:
                sf.write(p.source + '\n')
                tf.write(p.target + '\n')
    
    print(f"Saved {len(pairs)} pairs to {output_dir}")

In [ ]:
# Save all data
for pair in LANGUAGE_PAIRS:
    key = f"{pair['source']}-{pair['target']}"
    output_dir = DATA_DIR / key
    save_parallel_files(combined_data.get(key, []), output_dir, 'seed')

In [ ]:
# Save statistics
stats_file = DATA_DIR / 'corpus_stats.json'
with open(stats_file, 'w') as f:
    json.dump(all_stats, f, indent=2)
print(f"Saved statistics to {stats_file}")

## Summary

In [ ]:
print("=" * 50)
print("SEED DATA COLLECTION COMPLETE")
print("=" * 50)

total_pairs = sum(len(v) for v in combined_data.values())
print(f"\nTotal parallel pairs collected: {total_pairs:,}")

print("\nPer language pair:")
for pair in LANGUAGE_PAIRS:
    key = f"{pair['source']}-{pair['target']}"
    count = len(combined_data.get(key, []))
    print(f"  {pair['name']}: {count:,} pairs")

print(f"\nData saved to: {DATA_DIR.resolve()}")
print("\nNext step: Run 02_synthetic_corpus_generation.ipynb")